# 01 · Data acquisition & cleaning
**Project:** Urban PM2.5 / AQI forecasting for Delhi (SARIMAX, VAR & Granger causality)

| Source | What | Period |
|---|---|---|
| CPCB via Kaggle `rohanrao/air-quality-data-in-india` → `city_day.csv` | Daily PM2.5, NO₂, SO₂, AQI for Delhi | 2015-01-01 → 2020-07-01 |
| Open-Meteo Historical API (ERA5 reanalysis), 28.61°N 77.21°E | Daily mean relative humidity, wind speed (10 m), temperature, precipitation | same |

To get the CPCB file:
```
kaggle datasets download rohanrao/air-quality-data-in-india -f city_day.csv -p data/raw && unzip data/raw/city_day.csv.zip -d data/raw
```
The weather file downloads automatically below (no API key needed) and is cached in `data/raw/`.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data import (RAW, PROCESSED, RESULTS, DELHI, load_cpcb_city, fetch_openmeteo,
                      load_openmeteo, missing_report, clean_daily, to_weekly)

PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(exist_ok=True)
pd.set_option("display.precision", 2)

## 1. Load CPCB air-quality data (Delhi)

In [2]:
cpcb = load_cpcb_city(RAW / "city_day.csv", city="Delhi")
print(f"{len(cpcb)} days: {cpcb.index.min().date()} → {cpcb.index.max().date()}")
cpcb.head()

2009 days: 2015-01-01 → 2020-07-01


,pm25,no2,so2,aqi,aqi_bucket
Date,,,,,
2015-01-01,313.22,36.39,9.25,472.0,Severe
2015-01-02,186.18,32.87,6.65,454.0,Severe
2015-01-03,87.18,30.31,2.65,143.0,Moderate
2015-01-04,151.84,36.91,4.63,319.0,Very Poor
2015-01-05,146.60,34.92,3.33,325.0,Very Poor


## 2. Load weather (Open-Meteo ERA5) for the same dates

In [3]:
start, end = cpcb.index.min().date().isoformat(), cpcb.index.max().date().isoformat()
wx_path = fetch_openmeteo(RAW / "delhi_weather_openmeteo.json", start, end, **DELHI)
weather = load_openmeteo(wx_path)
weather.head()

,rh,wind,temp,precip
Date,,,,
2015-01-01,68,6.6,16.3,0.0
2015-01-02,83,7.0,15.2,4.2
2015-01-03,84,5.1,15.6,0.1
2015-01-04,78,10.2,14.4,0.0
2015-01-05,74,18.7,14.3,0.0


## 3. Merge on date and check missingness

In [4]:
daily_raw = cpcb.join(weather, how="outer")
assert len(daily_raw) == len(cpcb) == len(weather), "date ranges do not line up"
assert daily_raw.index.to_series().diff().dropna().eq(pd.Timedelta("1D")).all(), "dates are not continuous"

report = missing_report(daily_raw)
report.to_csv(RESULTS / "missing_report.csv")
report

,n_missing,pct_missing,longest_gap_days
pm25,2,0.10,2
no2,2,0.10,2
so2,110,5.48,81
aqi,10,0.50,4
rh,0,0.00,0
wind,0,0.00,0
temp,0,0.00,0
precip,0,0.00,0


**Cleaning rule:** gaps of **≤ 3 days** are filled by time interpolation. Longer gaps stay missing.
Only SO₂ has long gaps (28 and 81 days), and SO₂ is used only for descriptive EDA, not for modelling.
Negative concentrations are clipped to 0. PM2.5 is also log-transformed (`log_pm25`) to stabilise the
much larger winter variance.

In [5]:
daily = clean_daily(daily_raw, max_gap=3)
print("Days with ≥1 imputed value:", int(daily["imputed"].sum()))
missing_report(daily)

Days with ≥1 imputed value: 3


,n_missing,pct_missing,longest_gap_days
pm25,0,0.00,0
no2,0,0.00,0
so2,109,5.43,81
aqi,10,0.50,4
rh,0,0.00,0
wind,0,0.00,0
temp,0,0.00,0
precip,0,0.00,0
log_pm25,0,0.00,0


## 4. Weekly aggregation (for forecasting models)
Weeks end on Sunday. Weeks with fewer than 4 PM2.5 days (the partial last week) are dropped.
The weekly series has seasonal period **s = 52**, so a 12-month forecast is 52 steps ahead.

In [6]:
weekly = to_weekly(daily)
print(f"{len(weekly)} weeks: {weekly.index.min().date()} → {weekly.index.max().date()}")
weekly.head()

287 weeks: 2015-01-04 → 2020-06-28


,pm25,no2,so2,aqi,rh,wind,temp,precip,n_days,log_pm25
Date,,,,,,,,,,
2015-01-04,184.61,34.12,5.79,347.00,78.25,7.22,15.38,4.3,4,5.22
2015-01-11,196.01,41.19,4.44,358.43,79.43,9.91,13.17,0.0,7,5.28
2015-01-18,187.65,49.24,5.45,360.00,82.29,7.53,12.60,4.1,7,5.23
2015-01-25,156.36,24.63,6.52,326.14,80.00,8.09,13.34,20.8,7,5.05
2015-02-01,162.78,37.28,6.47,324.14,70.00,10.09,12.44,0.0,7,5.09


In [7]:
daily.to_csv(PROCESSED / "delhi_daily.csv")
weekly.to_csv(PROCESSED / "delhi_weekly.csv")

summary = pd.DataFrame({
    "daily": [len(daily), daily.index.min().date(), daily.index.max().date(), daily["pm25"].isna().sum()],
    "weekly": [len(weekly), weekly.index.min().date(), weekly.index.max().date(), weekly["pm25"].isna().sum()],
}, index=["rows", "start", "end", "PM2.5 missing after cleaning"])
summary

,daily,weekly
rows,2009,287
start,2015-01-01,2015-01-04
end,2020-07-01,2020-06-28
PM2.5 missing after cleaning,0,0
